# 1. Generate and prepare the modelling sample

This notebook contains the complete sample-development process: synthetic source generation, exact account linkage, data-quality exclusions, twelve-month outcome construction, treatment of intermediate cases, and the chronological development/holdout split.

Run this notebook before the three modelling notebooks. It writes the prepared data into the existing `data/Original data`, `data/Data Cleaning`, and `data/KGB Modelling` folders.

## Generate the synthetic source data

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

project_root = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "src" / "origination_scorecard" / "generator.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the Origination-Scorecard project")

source_path = str(project_root / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)

from origination_scorecard import GeneratorConfig, generate_project_data

config = GeneratorConfig(
    n_accounts=10_000,
    random_seed=20_260_907,
    target_known_bad_rate=0.04,
    target_inferred_rate=0.02,
    broken_link_rate=0.01,
)
outputs = generate_project_data(config)

original_dir = project_root / "data" / "Original data"
original_dir.mkdir(parents=True, exist_ok=True)
for name, frame in outputs.items():
    frame.to_csv(original_dir / f"{name}.csv", index=False, lineterminator="\n")

display(pd.DataFrame([
    {"Source": name, "Rows": len(frame), "Columns": len(frame.columns)}
    for name, frame in outputs.items()
]))

,Source,Rows,Columns
0,applications,9987,38
1,performance_12m,10012,13
2,experian_consumer,10000,15
3,experian_consumer_dictionary,15,5


### Synthetic-data design

The generator produces fabricated application, bureau and twelve-month performance records. Approximately 4% of accounts have three or more missed-payment months and 2% have one or two. Risk is partly driven by utilisation, searches, historic defaults and arrears, employment/address instability, low disposable income and high DTI, plus random noise. One per cent of links are deliberately damaged for the linkage exercise.

This supports a reproducible methodology demonstration. It does not establish real lending performance, fairness, regulatory compliance or commercial value.

## Clean and link the three sources

In [2]:
applications = pd.read_csv(original_dir / "applications.csv", dtype={"account_id": "string"})
performance = pd.read_csv(original_dir / "performance_12m.csv", dtype={"account_id": "string"})
delphi_raw = pd.read_csv(original_dir / "experian_consumer.csv", dtype={"account_id": "string"})

INPUT_ID_PATTERN = r"ACC(?:\s*-\s*|\s+)([0-9]{8})"
CANONICAL_ID_PATTERN = r"ACC-[0-9]{8}"
payment_columns = [f"missed_payment_m{month:02d}" for month in range(1, 13)]

def normalise_account_ids(values):
    text = values.astype("string").str.strip().str.upper()
    digits = text.str.extract(rf"^{INPUT_ID_PATTERN}$", expand=False)
    return ("ACC-" + digits).astype("string")

def clean_source(data, source_name, invalid_extra=None, invalid_reason=None):
    rows = data.copy()
    rows["original_account_id"] = rows["account_id"]
    canonical = normalise_account_ids(rows["original_account_id"])
    raw = rows["original_account_id"]
    missing = raw.isna() | raw.str.strip().eq("").fillna(False)
    invalid = ~missing & canonical.isna()
    duplicate = canonical.notna() & canonical.duplicated(keep=False)
    rows["account_id"] = canonical

    reason = pd.Series(pd.NA, index=rows.index, dtype="string")
    reason.loc[missing] = "missing account_id"
    reason.loc[invalid] = "invalid account_id format"
    reason.loc[duplicate] = "duplicate account_id after normalisation"
    if invalid_extra is not None:
        reason.loc[reason.isna() & invalid_extra] = invalid_reason

    exceptions = rows.loc[reason.notna()].copy()
    exceptions.insert(0, "source_row_number", exceptions.index + 2)
    exceptions.insert(0, "exception_reason", reason.loc[exceptions.index])
    exceptions.insert(0, "source_table", source_name)
    clean = rows.loc[reason.isna()].copy()
    return clean, exceptions

missing_payment_columns = sorted(set(payment_columns) - set(performance.columns))
if missing_payment_columns:
    raise ValueError(f"Missing performance columns: {missing_payment_columns}")
invalid_performance = ~performance[payment_columns].isin([0, 1]).all(axis=1)

if "DelphiScore" not in delphi_raw.columns:
    raise ValueError("DelphiScore is missing")
delphi = delphi_raw[["account_id", "DelphiScore"]].copy()
invalid_delphi = pd.to_numeric(delphi["DelphiScore"], errors="coerce").isna()

applications_clean, application_exceptions = clean_source(applications, "applications")
performance_clean, performance_exceptions = clean_source(
    performance,
    "performance_12m",
    invalid_performance,
    "invalid or missing monthly performance value",
)
delphi_clean, delphi_exceptions = clean_source(
    delphi,
    "experian_consumer",
    invalid_delphi,
    "missing or non-numeric DelphiScore",
)
delphi_clean["DelphiScore"] = pd.to_numeric(delphi_clean["DelphiScore"], errors="raise")

clean_tables = {
    "applications": applications_clean,
    "performance_12m": performance_clean,
    "experian_consumer": delphi_clean,
}
id_sets = {name: set(rows["account_id"]) for name, rows in clean_tables.items()}
common_ids = set.intersection(*id_sets.values())

exception_parts = [application_exceptions, performance_exceptions, delphi_exceptions]
for source_name, rows in clean_tables.items():
    other_sets = {name: ids for name, ids in id_sets.items() if name != source_name}
    reason = pd.Series(
        [
            "no exact match in " + " and ".join(
                name for name, ids in other_sets.items() if account_id not in ids
            )
            if any(account_id not in ids for ids in other_sets.values()) else pd.NA
            for account_id in rows["account_id"]
        ],
        index=rows.index,
        dtype="string",
    )
    unmatched = rows.loc[reason.notna()].copy()
    unmatched.insert(0, "source_row_number", unmatched.index + 2)
    unmatched.insert(0, "exception_reason", reason.loc[unmatched.index])
    unmatched.insert(0, "source_table", source_name)
    exception_parts.append(unmatched)

joined = {
    name: rows.loc[rows["account_id"].isin(common_ids)].drop(columns="original_account_id")
    for name, rows in clean_tables.items()
}
working_data = (
    joined["applications"]
    .merge(joined["performance_12m"], on="account_id", validate="one_to_one")
    .merge(joined["experian_consumer"], on="account_id", validate="one_to_one")
    .sort_values("account_id")
    .reset_index(drop=True)
)
linkage_exceptions = pd.concat(exception_parts, ignore_index=True, sort=False)

if not working_data["account_id"].is_unique:
    raise AssertionError("Linked account IDs are not unique")
if not working_data["account_id"].str.fullmatch(CANONICAL_ID_PATTERN).all():
    raise AssertionError("A non-canonical account ID entered the linked data")
if not working_data[payment_columns].isin([0, 1]).all().all():
    raise AssertionError("Invalid performance values entered the linked data")

clean_dir = project_root / "data" / "Data Cleaning"
clean_dir.mkdir(parents=True, exist_ok=True)
working_data.to_csv(clean_dir / "clean_data.csv", index=False, lineterminator="\n")
linkage_exceptions.to_csv(
    clean_dir / "linkage_exceptions.csv", index=False, lineterminator="\n"
)

print(f"Linked accounts retained: {len(working_data):,}")
print(f"Excluded source rows: {len(linkage_exceptions):,}")
display(linkage_exceptions.groupby(["source_table", "exception_reason"]).size().rename("Rows"))

Linked accounts retained: 9,913
Excluded source rows: 260


source_table       exception_reason                        
applications       duplicate account_id after normalisation    24
                   no exact match in performance_12m           50
experian_consumer  no exact match in applications              37
                   no exact match in performance_12m           50
performance_12m    invalid account_id format                   37
                   missing account_id                          25
                   no exact match in applications              37
Name: Rows, dtype: int64

## Define outcomes and create the KGB population

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

project_root = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "data" / "Data Cleaning" / "clean_data.csv").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate data/Data Cleaning/clean_data.csv")

clean_path = project_root / "data" / "Data Cleaning" / "clean_data.csv"
output_dir = project_root / "data" / "KGB Modelling"
output_dir.mkdir(parents=True, exist_ok=True)
working_data = pd.read_csv(clean_path)
print(f"Loaded {len(working_data):,} linked accounts from {clean_path}")

Loaded 9,913 linked accounts from c:\Users\georg\OneDrive\Coding\Origination-Scorecard\data\Data Cleaning\clean_data.csv


## Validate the twelve-month performance window

In [4]:
payment_columns = [f"missed_payment_m{month:02d}" for month in range(1, 13)]
required = ["account_id", *payment_columns]
missing = sorted(set(required) - set(working_data.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")
if working_data["account_id"].isna().any() or working_data["account_id"].duplicated().any():
    raise ValueError("account_id must be complete and unique")

payments = working_data[payment_columns].apply(pd.to_numeric, errors="coerce")
valid_performance = payments.isin([0, 1]).all(axis=1)
exceptions = working_data.loc[~valid_performance].copy()
exceptions["invalid_payment_data"] = 1
exceptions["exception_reason"] = "Missing or invalid monthly performance value"
exceptions.to_csv(
    project_root / "data" / "Data Cleaning" / "exceptions.csv",
    index=False,
    lineterminator="\n",
)

working_data = working_data.loc[valid_performance].copy()
payments = payments.loc[valid_performance].astype(int)
working_data[payment_columns] = payments
working_data["invalid_payment_data"] = 0
working_data["missed_payment_months"] = payments.sum(axis=1).astype(int)

print(f"Valid accounts: {len(working_data):,}")
print(f"Performance exceptions: {len(exceptions):,}")

Valid accounts: 9,913
Performance exceptions: 0


## Create descriptive outcomes and the KGB target

In [5]:
missed = working_data["missed_payment_months"]
working_data["outcome"] = np.select(
    [missed.eq(0), missed.between(1, 2), missed.ge(3)],
    ["Known Good", "Intermediate", "Known Bad"],
    default="Invalid",
)
if working_data["outcome"].eq("Invalid").any():
    raise ValueError("A missed-payment total falls outside the target definitions")

# Nullable integer: intermediate rows remain unlabelled for KGB modelling.
working_data["target_bad"] = pd.Series(pd.NA, index=working_data.index, dtype="Int64")
working_data.loc[working_data["outcome"].eq("Known Good"), "target_bad"] = 0
working_data.loc[working_data["outcome"].eq("Known Bad"), "target_bad"] = 1

treatment = {
    "Known Good": "Included in KGB as 0",
    "Intermediate": "Excluded from KGB; reintroduced in KIGB as kigb_bad = 0",
    "Known Bad": "Included in KGB as 1",
}
breakdown = (
    working_data.groupby("outcome", observed=True)
    .agg(Accounts=("account_id", "size"), KGB_target_non_missing=("target_bad", "count"))
    .reindex(["Known Good", "Intermediate", "Known Bad"])
    .reset_index()
)
breakdown["Share"] = breakdown["Accounts"] / breakdown["Accounts"].sum()
breakdown["Model treatment"] = breakdown["outcome"].map(treatment)
display(breakdown.style.format({"Accounts": "{:,.0f}", "Share": "{:.2%}"}))

,outcome,Accounts,KGB_target_non_missing,Share,Model treatment
0,Known Good,"9,320",9320,94.02%,Included in KGB as 0
1,Intermediate,197,0,1.99%,Excluded from KGB; reintroduced in KIGB as kigb_bad = 0
2,Known Bad,396,396,3.99%,Included in KGB as 1


## Export the intermediate and KGB populations

In [6]:
indeterminates = working_data.loc[working_data["outcome"].eq("Intermediate")].copy()
kgb_model = working_data.loc[working_data["outcome"].isin(["Known Good", "Known Bad"])].copy()
kgb_model["target_bad"] = kgb_model["target_bad"].astype(int)

if indeterminates["target_bad"].notna().any():
    raise AssertionError("Intermediate KGB targets must remain missing")
if not kgb_model["target_bad"].isin([0, 1]).all():
    raise AssertionError("KGB target must contain only 0 and 1")

indeterminate_path = output_dir / "indeterminates.csv"
kgb_path = output_dir / "KGB model.csv"
indeterminates.to_csv(indeterminate_path, index=False, lineterminator="\n")
kgb_model.to_csv(kgb_path, index=False, lineterminator="\n")

print(f"Intermediate rows: {len(indeterminates):,} -> {indeterminate_path}")
print(f"KGB rows: {len(kgb_model):,} -> {kgb_path}")

Intermediate rows: 197 -> c:\Users\georg\OneDrive\Coding\Origination-Scorecard\data\KGB Modelling\indeterminates.csv
KGB rows: 9,716 -> c:\Users\georg\OneDrive\Coding\Origination-Scorecard\data\KGB Modelling\KGB model.csv


## Create the chronological development and holdout samples

In [7]:
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "data" / "KGB Modelling" / "KGB model.csv").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the KGB model dataset")

data_dir = project_root / "data" / "KGB Modelling"
model_data = pd.read_csv(data_dir / "KGB model.csv")
model_data["application_date"] = pd.to_datetime(model_data["application_date"], errors="raise")

if not model_data["target_bad"].isin([0, 1]).all():
    raise ValueError("KGB target must contain only known 0/1 labels")
if model_data["outcome"].eq("Intermediate").any():
    raise ValueError("Intermediate rows must not enter KGB sampling")

## Apply and verify the date boundary

In [8]:
split_date = pd.Timestamp("2024-07-01")
build_data = model_data.loc[model_data["application_date"] < split_date].copy()
holdout_data = model_data.loc[model_data["application_date"] >= split_date].copy()

if set(build_data["account_id"]) & set(holdout_data["account_id"]):
    raise ValueError("An account appears in both samples")
if len(build_data) + len(holdout_data) != len(model_data):
    raise ValueError("The split lost or duplicated accounts")

def sample_row(name, data, method):
    return {
        "Sample": name,
        "Method": method,
        "Start date": data["application_date"].min().date(),
        "End date": data["application_date"].max().date(),
        "Accounts": len(data),
        "Known goods": int(data["target_bad"].eq(0).sum()),
        "Known bads": int(data["target_bad"].eq(1).sum()),
        "Bad rate": data["target_bad"].mean(),
    }

split_summary = pd.DataFrame([
    sample_row("Development", build_data, "Application date < 2024-07-01"),
    sample_row("Chronological holdout", holdout_data, "Application date >= 2024-07-01"),
])
display(split_summary.style.format({"Accounts": "{:,.0f}", "Bad rate": "{:.2%}"}))

,Sample,Method,Start date,End date,Accounts,Known goods,Known bads,Bad rate
0,Development,Application date < 2024-07-01,2023-01-01,2024-06-30,"7,258",6964,294,4.05%
1,Chronological holdout,Application date >= 2024-07-01,2024-07-01,2024-12-30,"2,458",2356,102,4.15%


## Application-time predictor and leakage audit

The final reduced scorecard only uses information that is available when the application is assessed. The audit below documents the source and timing. Twelve-month performance, target labels and outputs created by the model are prohibited inputs.

In [9]:
leakage_audit = pd.DataFrame([
    ("debt_to_income_ratio", "Application income and committed outgoings", "At application", "Use"),
    ("recent_application_count", "Application / bureau search history", "At application", "Use"),
    ("revolving_utilisation", "Bureau snapshot", "At application", "Use"),
    ("historic_defaults", "Bureau history before this application", "At application", "Use"),
    ("log_address_tenure", "Derived from stated address tenure", "At application", "Use"),
    ("low_disposable_income_flag", "Derived from stated income and outgoings", "At application", "Use"),
    ("loan_purpose=education", "Application purpose", "At application", "Use"),
    ("DelphiScore", "Bureau score snapshot", "At application", "Candidate comparison only"),
    ("missed_payment_m01...m12", "Performance after origination", "After decision", "Exclude"),
    ("missed_payment_months / target_bad / kigb_bad", "Outcome construction", "After decision", "Exclude"),
    ("predicted_pd / score / decision", "Model outputs", "Created during modelling", "Exclude"),
    ("name, address, postcode, date of birth, IDs", "PII / identifiers", "At application", "Exclude"),
], columns=["Field", "Source", "Availability", "Treatment"])
display(leakage_audit)

prohibited = {
    "missed_payment_months", "target_bad", "kigb_bad", "predicted_pd", "score", "decision",
    *[f"missed_payment_m{month:02d}" for month in range(1, 13)],
}
final_predictors = {
    "debt_to_income_ratio", "recent_application_count", "revolving_utilisation",
    "historic_defaults", "log_address_tenure", "low_disposable_income_flag",
    "loan_purpose=education",
}
if prohibited & final_predictors:
    raise AssertionError("A prohibited post-outcome field entered the final predictor list")

,Field,Source,Availability,Treatment
0,debt_to_income_ratio,Application income and committed outgoings,At application,Use
1,recent_application_count,Application / bureau search history,At application,Use
2,revolving_utilisation,Bureau snapshot,At application,Use
3,historic_defaults,Bureau history before this application,At application,Use
4,log_address_tenure,Derived from stated address tenure,At application,Use
5,low_disposable_income_flag,Derived from stated income and outgoings,At application,Use
6,loan_purpose=education,Application purpose,At application,Use
7,DelphiScore,Bureau score snapshot,At application,Candidate comparison only
8,missed_payment_m01...m12,Performance after origination,After decision,Exclude
9,missed_payment_months / target_bad / kigb_bad,Outcome construction,After decision,Exclude


## Save the development and holdout samples

In [10]:
in_path = data_dir / "KGB in sample.csv"
out_path = data_dir / "KGB out sample.csv"
build_data["application_date"] = build_data["application_date"].dt.strftime("%Y-%m-%d")
holdout_data["application_date"] = holdout_data["application_date"].dt.strftime("%Y-%m-%d")
build_data.to_csv(in_path, index=False, lineterminator="\n")
holdout_data.to_csv(out_path, index=False, lineterminator="\n")
print(f"Development: {len(build_data):,} -> {in_path}")
print(f"Chronological holdout: {len(holdout_data):,} -> {out_path}")

Development: 7,258 -> c:\Users\georg\OneDrive\Coding\Origination-Scorecard\data\KGB Modelling\KGB in sample.csv
Chronological holdout: 2,458 -> c:\Users\georg\OneDrive\Coding\Origination-Scorecard\data\KGB Modelling\KGB out sample.csv
